# Rango de Frecuencias

En este notebook se buscará predecir la variable objetivo realizando el preprocesamiento de los datos agrupando por frecuencias para así reducir la dimensión. Se usará leave one out cross validation para encontrar los parámetros.

## Bibliotecas

In [3]:
library(glmnet)
library(pracma)
source("Utils.R")

In [4]:
vasijas_X <- read.csv("Vessel_X.txt", header = FALSE)
vasijas_Y <- read.csv("Vessel_Y.txt", header = FALSE)
oxido_sodio_Y <- c(vasijas_Y$V1)

merged_vasijas <- vasijas_X
merged_vasijas$Y <- oxido_sodio_Y

# Dividimos en set de train y set de hold out

In [5]:
set.seed(5)
hold_out_ind <- sample(seq_len(nrow(vasijas_X)), size = 20)

HOLDOUT_X <- vasijas_X[hold_out_ind, ]
HOLDOUT_Y <- oxido_sodio_Y[hold_out_ind]
TRAIN_Y <- oxido_sodio_Y[-hold_out_ind]
TRAIN_X <- vasijas_X[-hold_out_ind, ]

# Funcion para means

In [6]:
# Función que recibe un dataframe ('X') y un tamanio de ventana ('tamanio_ventana', 'tv')
# Devuelve otro dataframe donde cada columna es la media de las vecinas
#  en ventanas del tamanio recibido como segundo parametro.

# Por ejemplo:

# X=                       Y=
# A B C D E F G            Y1  Y2  Y3  Y4
# 1 2 1 2 1 2 1    tv=2    1.5 1.5 1.5 1.0
# 2 2 2 2 2 2 2    --->    2.0 2.0 2.0 2.0
# 3 3 4 4 5 5 1            3.0 4.0 5.0 1.0
# 1 2 3 4 5 6 7            1.5 3.5 5.5 7.0

preprocesamiento_agrupar <- function(X, tamanio_ventana) {
  X_means <- data.frame(matrix(NA, nrow = nrow(X), ncol = ceiling(ncol(X) / tamanio_ventana)))
  last_i <- floor(ncol(X) / tamanio_ventana)
  for (i in 1:last_i) {
    j <- i - 1
    X_means[,i] <- rowMeans(X[, (j * tamanio_ventana + 1): (i * tamanio_ventana)])
  }
  num_last_cols <- i * tamanio_ventana + ncol(X) %% tamanio_ventana
  num_first_last_cols <- last_i * tamanio_ventana + 1
  if(num_first_last_cols > num_last_cols){
      return(X_means)
  } else if(num_first_last_cols != num_last_cols) {
    X_means[,ceiling(ncol(X) / tamanio_ventana)] <- rowMeans(X[, num_first_last_cols: num_last_cols])
  } else {
    X_means[,ceiling(ncol(X) / tamanio_ventana)] <- X[, num_first_last_cols]
  }
  return (X_means)
}

In [ ]:
#corr_matrix <- cor(X_means_train, method = "pearson")
#corr_matrix

In [7]:
set.seed(5)

n = nrow(TRAIN_X)
k = 5
resultados <- data.frame(
  Ventana =integer(), 
  Alpha   =double(),
  Lambda  =double(),
  Error   =double()
)

indices = obtener_indices_kfold(n, k)

for(ventana in seq(2, 30, 1)){
    for(alpha in seq(0, 1, 0.1)){
        for(lambda in logspace(-10, -2, 9)){
            err = 0
            for(i in indices){
                X <- preprocesamiento_agrupar(TRAIN_X, ventana)
                
                X_train_scaled = scale(X[-i,])
                Y_train = TRAIN_Y[-i]
                
                X_test_scaled = scale(X[i,], center=attr(X_train_scaled, "scaled:center"),
                              scale=attr(X_train_scaled, "scaled:scale"))

                Y_test = TRAIN_Y[i]

                model <- glmnet(X_train_scaled, Y_train, lambda=lambda, alpha=alpha)
                Y_pred = predict(model, X_test_scaled)
                err = err + sum((Y_test - Y_pred)^2)
            }
            resultados = rbind(resultados, list(ventana, alpha, lambda, err/n))
        }
    }
    message(progreso(ventana, seq(2, 30, 1)))
}
colnames(resultados)  <- c("Ventana", "alpha", "lambda", "Error")

El progreso es del 3.4 %

El progreso es del 6.9 %

El progreso es del 10.3 %

El progreso es del 13.8 %

El progreso es del 17.2 %

El progreso es del 20.7 %

El progreso es del 24.1 %

El progreso es del 27.6 %

El progreso es del 31 %

El progreso es del 34.5 %

El progreso es del 37.9 %

El progreso es del 41.4 %

El progreso es del 44.8 %

El progreso es del 48.3 %

El progreso es del 51.7 %

El progreso es del 55.2 %

El progreso es del 58.6 %

El progreso es del 62.1 %

El progreso es del 65.5 %

El progreso es del 69 %

El progreso es del 72.4 %

El progreso es del 75.9 %

El progreso es del 79.3 %

El progreso es del 82.8 %

El progreso es del 86.2 %

El progreso es del 89.7 %

El progreso es del 93.1 %

El progreso es del 96.6 %

El progreso es del 100 %



In [8]:
resultados

,Ventana,alpha,lambda,Error
,<dbl>,<dbl>,<dbl>,<dbl>
1,2,0.0,1e-10,2.2560932
2,2,0.0,1e-09,2.2560927
3,2,0.0,1e-08,2.2560877
4,2,0.0,1e-07,2.2560378
5,2,0.0,1e-06,2.2553682
6,2,0.0,1e-05,2.2492837
7,2,0.0,1e-04,2.1906315
8,2,0.0,1e-03,1.8295913
9,2,0.0,1e-02,1.1808320


In [9]:
which.min(resultados$Error)

[1] 1294

In [10]:
resultados[which.min(resultados$Error),]

,Ventana,alpha,lambda,Error
,<dbl>,<dbl>,<dbl>,<dbl>
1294,15,0,1e-04,0.6302542


In [11]:

modelo = glmnet(ventana=8, alpha=1, lambda=1e-05)


ERROR: Error in glmnet(ventana = 8, alpha = 1, lambda = 1e-05): el argumento "x" está ausente, sin valor por omisión
